In [ ]:
!pip install catboost

# Initial

In [ ]:
# ─────────────────────────────────────────────────────────────
# STANDARD LIBRARY
# ─────────────────────────────────────────────────────────────
import os
import sys
import math
import time
import warnings
import logging
from math import radians, sin, cos, sqrt, atan2
from typing import Tuple, List, Dict, Optional, Union

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

# ─────────────────────────────────────────────────────────────
# DATA
# ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from pandas import DataFrame, Series

# ─────────────────────────────────────────────────────────────
# SKLEARN — preprocessing, splitting, metrics
# ─────────────────────────────────────────────────────────────
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
)
from sklearn.preprocessing import (
    LabelEncoder,
    OrdinalEncoder,
    StandardScaler,
    MinMaxScaler,
)
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,   # AUC-PR — better metric for imbalanced data
    precision_recall_curve,
    confusion_matrix,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.pipeline import Pipeline

# ─────────────────────────────────────────────────────────────
# IMBALANCED-LEARN — SMOTE and variants
# pip install imbalanced-learn
# ─────────────────────────────────────────────────────────────
from imblearn.over_sampling import (
    SMOTE,
    SMOTENC,      # SMOTE for datasets with categorical columns
    BorderlineSMOTE,
    SVMSMOTE,
)
from imblearn.combine import SMOTETomek   # SMOTE + Tomek undersampling
from imblearn.pipeline import Pipeline as ImbPipeline

# ─────────────────────────────────────────────────────────────
# XGBOOST
# pip install xgboost
# ─────────────────────────────────────────────────────────────
import xgboost as xgb
from xgboost import XGBClassifier

# ─────────────────────────────────────────────────────────────
# LIGHTGBM
# pip install lightgbm
# ─────────────────────────────────────────────────────────────
import lightgbm as lgb
from lightgbm import LGBMClassifier

# ─────────────────────────────────────────────────────────────
# CATBOOST
# pip install catboost
# ─────────────────────────────────────────────────────────────
from catboost import CatBoostClassifier, Pool

# ─────────────────────────────────────────────────────────────
# SHAP — feature importance & explainability
# pip install shap
# ─────────────────────────────────────────────────────────────
import shap

# ─────────────────────────────────────────────────────────────
# VISUALISATION
# pip install matplotlib seaborn
# ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (10, 5)})

In [ ]:
df = pd.read_csv("hf://datasets/dazzle-nu/CIS435-CreditCardFraudDetection/fraudTrain.csv")

In [ ]:
df = df.drop(columns=["Unnamed: 0", "Unnamed: 23", "6006"], errors='ignore')

In [ ]:
df.head()

Credit Card Fraud Detection - Feature Engineering Pipeline

Combines standard fraud detection features WITH cdeotte's "Magic" techniques
adapted from: https://www.kaggle.com/cdeotte/xgb-fraud-with-magic-0-9600

The "Magic" consists of two parts (adapted to this dataset):
  1. UID — a unique client identifier built from stable cardholder attributes
            (cc_num + zip + dob) that ties all transactions to the same person,
            even if cc_num alone is noisy or shared.
  2. UID Aggregations — group stats computed per UID across the full dataset
            (mean/std of amt, fraud rate per UID, etc.) which act as
            client-level behavioral signatures the model can exploit.
  3. Post-processing — after model prediction, replace each UID's predictions
            with the UID's mean prediction (all txns from same client should
            have similar fraud probability). See postprocess_predictions().

Usage:
    X_engineered = engineer_all_features(X, y, fit=True)   # on training data
    X_engineered = engineer_all_features(X, y=None, fit=False)  # on test/val

    fit=True  → computes and stores all stats (merchant, card, UID aggregates)
    fit=False → uses stored stats from training (prevents leakage)

Input:  X DataFrame (all columns except is_fraud)
        y Series    (is_fraud — required when fit=True for target encoding)
Output: X DataFrame with all engineered features, raw columns dropped

In [ ]:
_stats = {}

# Helper Functions

In [ ]:
# 0. PREPROCESSING — parse types, fix formats
def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    """Parse datetime, dob, and ensure correct dtypes."""
    df = df.copy()

    # Parse transaction datetime
    df["trans_datetime"] = pd.to_datetime(
        df["trans_date_trans_time"], infer_datetime_format=True
    )

    # Parse date of birth
    df["dob_parsed"] = pd.to_datetime(df["dob"], infer_datetime_format=True)

    # Ensure amount is float
    df["amt"] = df["amt"].astype(float)

    # Sort by card number then time (critical for velocity features)
    df = df.sort_values(["cc_num", "unix_time"]).reset_index(drop=True)

    return df


In [ ]:
# 1. TEMPORAL FEATURES
def add_temporal_features(df: pd.DataFrame) -> pd.DataFrame:
    """Extract time-based features from transaction datetime."""
    df = df.copy()
    dt = df["trans_datetime"]

    df["hour"]             = dt.dt.hour
    df["day_of_week"]      = dt.dt.dayofweek          # 0=Monday, 6=Sunday
    df["day_of_month"]     = dt.dt.day
    df["month"]            = dt.dt.month
    df["quarter"]          = dt.dt.quarter
    df["is_weekend"]       = (df["day_of_week"] >= 5).astype(int)
    df["is_night"]         = ((df["hour"] >= 23) | (df["hour"] <= 4)).astype(int)
    df["is_business_hours"]= ((df["hour"] >= 9) & (df["hour"] <= 17)
                               & (df["is_weekend"] == 0)).astype(int)

    # Cyclic encoding for hour and day (preserves circular continuity)
    df["hour_sin"]         = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"]         = np.cos(2 * np.pi * df["hour"] / 24)
    df["dow_sin"]          = np.sin(2 * np.pi * df["day_of_week"] / 7)
    df["dow_cos"]          = np.cos(2 * np.pi * df["day_of_week"] / 7)

    return df

In [ ]:
# 2. CARDHOLDER AGE FEATURES
def add_age_features(df: pd.DataFrame) -> pd.DataFrame:
    """Compute age at time of transaction and age group flags."""
    df = df.copy()

    df["age"] = (
        (df["trans_datetime"] - df["dob_parsed"]).dt.days / 365.25
    ).astype(int)

    df["age_group"] = pd.cut(
        df["age"],
        bins=[0, 30, 45, 60, 200],
        labels=["18-30", "31-45", "46-60", "60+"]
    ).astype(str)

    df["is_senior"] = (df["age"] >= 65).astype(int)

    return df

In [ ]:
# 3. GEOGRAPHIC / DISTANCE FEATURES
def _haversine_vectorized(lat1, lon1, lat2, lon2) -> np.ndarray:
    """Compute haversine distance (km) between two sets of lat/lon arrays."""
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return R * 2 * np.arcsin(np.sqrt(a))


def add_geographic_features(df: pd.DataFrame) -> pd.DataFrame:
    """Compute distance between cardholder home and merchant location."""
    df = df.copy()

    df["distance_from_home"] = _haversine_vectorized(
        df["lat"].values, df["long"].values,
        df["merch_lat"].values, df["merch_long"].values
    )

    df["log_distance"]      = np.log1p(df["distance_from_home"])
    df["is_far_transaction"]= (df["distance_from_home"] > 100).astype(int)

    # Raw directional deltas
    df["merch_lat_diff"]    = df["merch_lat"] - df["lat"]
    df["merch_long_diff"]   = df["merch_long"] - df["long"]

    # Log city population (highly skewed)
    df["log_city_pop"]      = np.log1p(df["city_pop"])

    return df

In [ ]:
# 4. AMOUNT FEATURES
def add_amount_features(df: pd.DataFrame) -> pd.DataFrame:
    """Amount transformations and anomaly flags."""
    df = df.copy()

    df["log_amt"]       = np.log1p(df["amt"])

    # Round number detection (card testing behavior)
    df["amt_round"]     = (df["amt"] % 1 == 0).astype(int)
    df["amt_cents_only"]= (df["amt"] < 1).astype(int)  # micro-transactions

    # Quantile bins
    df["amt_bin"] = pd.qcut(
        df["amt"],
        q=4,
        labels=["micro", "small", "medium", "large"],
        duplicates="drop"
    ).astype(str)

    df["is_high_value"] = (df["amt"] > df["amt"].quantile(0.95)).astype(int)

    return df

In [ ]:
# 5. VELOCITY / BEHAVIORAL FEATURES  (per card, time-ordered)
def add_velocity_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute per-card rolling behavioral features.
    IMPORTANT: df must be sorted by (cc_num, unix_time) before calling.
    """
    df = df.copy()

    # ── 5a. Time since last transaction (seconds) ──────────────
    df["time_since_last_txn"] = (
        df.groupby("cc_num")["unix_time"]
        .diff()
        .fillna(999999)   # first transaction gets a large placeholder
    )
    df["log_time_since_last"] = np.log1p(df["time_since_last_txn"])

    # ── 5b. Amount compared to previous transaction ─────────────
    df["amt_last_txn"]       = df.groupby("cc_num")["amt"].shift(1).fillna(0)
    df["amt_diff_from_last"] = df["amt"] - df["amt_last_txn"]

    # ── 5c. Rolling transaction counts (approximate via unix_time) ─
    # Using expanding window approach with groupby + apply
    def rolling_count(group, window_seconds):
        counts = []
        times = group["unix_time"].values
        for i, t in enumerate(times):
            count = np.sum((times[:i] >= t - window_seconds) & (times[:i] < t))
            counts.append(count)
        return pd.Series(counts, index=group.index)

    # For large datasets, use optimized vectorized approach
    df["txn_count_last_1h"]  = (
        df.groupby("cc_num", group_keys=False)
        .apply(lambda g: rolling_count(g, 3600))
    )
    df["txn_count_last_24h"] = (
        df.groupby("cc_num", group_keys=False)
        .apply(lambda g: rolling_count(g, 86400))
    )
    df["txn_count_last_7d"]  = (
        df.groupby("cc_num", group_keys=False)
        .apply(lambda g: rolling_count(g, 604800))
    )

    # ── 5d. Rolling amount statistics ──────────────────────────
    # 7-day rolling mean & std per card (using expanding as proxy for rolling by time)
    df["amt_mean_7d"] = (
        df.groupby("cc_num")["amt"]
        .transform(lambda x: x.expanding().mean().shift(1))
        .fillna(df["amt"])
    )
    df["amt_std_7d"] = (
        df.groupby("cc_num")["amt"]
        .transform(lambda x: x.expanding().std().shift(1))
        .fillna(1)
    )
    df["amt_zscore_card"] = (
        (df["amt"] - df["amt_mean_7d"]) / (df["amt_std_7d"] + 1e-6)
    )
    df["amt_vs_mean_7d"] = df["amt"] / (df["amt_mean_7d"] + 1e-6)

    # ── 5e. Unique merchant / category counts ──────────────────
    def rolling_unique(group, col, window_seconds):
        result = []
        times = group["unix_time"].values
        vals  = group[col].values
        for i, t in enumerate(times):
            mask = (times[:i] >= t - window_seconds) & (times[:i] < t)
            result.append(len(set(vals[:i][mask])))
        return pd.Series(result, index=group.index)

    df["unique_merchants_24h"] = (
        df.groupby("cc_num", group_keys=False)
        .apply(lambda g: rolling_unique(g, "merchant", 86400))
    )
    df["unique_categories_24h"] = (
        df.groupby("cc_num", group_keys=False)
        .apply(lambda g: rolling_unique(g, "category", 86400))
    )

    # ── 5f. New merchant / new category flags ──────────────────
    def is_new_entity(group, col):
        seen = set()
        flags = []
        for val in group[col]:
            flags.append(int(val not in seen))
            seen.add(val)
        return pd.Series(flags, index=group.index)

    df["is_new_merchant"] = (
        df.groupby("cc_num", group_keys=False)
        .apply(lambda g: is_new_entity(g, "merchant"))
    )
    df["is_new_category"] = (
        df.groupby("cc_num", group_keys=False)
        .apply(lambda g: is_new_entity(g, "category"))
    )

    return df

In [ ]:
# 6. CARDHOLDER PROFILE AGGREGATES  (fit/transform pattern)
def add_card_profile_features(df: pd.DataFrame, fit: bool = True) -> pd.DataFrame:
    """
    Aggregate cardholder-level behavioral profile.
    fit=True  → compute from current df and store in _stats
    fit=False → merge from stored _stats (no leakage)
    """
    df = df.copy()

    if fit:
        card_stats = df.groupby("cc_num").agg(
            card_avg_amt   = ("amt", "mean"),
            card_std_amt   = ("amt", "std"),
            card_max_amt   = ("amt", "max"),
            card_txn_count = ("amt", "count"),
            card_home_state= ("state", lambda x: x.mode()[0]),
            card_fav_cat   = ("category", lambda x: x.mode()[0]),
            card_avg_hour  = ("hour", "mean"),
        ).reset_index()
        _stats["card_stats"] = card_stats

    card_stats = _stats["card_stats"]
    df = df.merge(card_stats, on="cc_num", how="left")

    df["card_amt_zscore"]         = (df["amt"] - df["card_avg_amt"]) / (df["card_std_amt"] + 1e-6)
    df["is_above_card_max"]       = (df["amt"] > df["card_max_amt"]).astype(int)
    df["is_unusual_state"]        = (df["state"] != df["card_home_state"]).astype(int)
    df["is_unusual_category"]     = (df["category"] != df["card_fav_cat"]).astype(int)
    df["hour_deviation_from_norm"]= abs(df["hour"] - df["card_avg_hour"])

    return df

In [ ]:
# 7. MERCHANT AGGREGATE FEATURES  (fit/transform pattern)
def add_merchant_features(df: pd.DataFrame, y: pd.Series = None, fit: bool = True) -> pd.DataFrame:
    """
    Merchant-level aggregates including Bayesian-smoothed fraud rate.
    Formula: (fraud_count + K * global_rate) / (txn_count + K)
    Separates 'unknown risk' (new merchant) from 'known clean' (high txn count, 0 fraud).
    K=10. fit=True computes from training only — no leakage.
    """
    df = df.copy()
    if fit:
        if y is None:
            raise ValueError("y required when fit=True")
        GLOBAL_RATE = y.mean()
        K = 10
        temp = df.copy(); temp["is_fraud"] = y.values
        ms = temp.groupby("merchant").agg(
            merchant_avg_amt    = ("amt",      "mean"),
            merchant_std_amt    = ("amt",      "std"),
            merchant_txn_count  = ("amt",      "count"),
            _fraud_sum          = ("is_fraud", "sum"),
        ).reset_index()
        ms["merchant_fraud_rate"] = (ms["_fraud_sum"] + K * GLOBAL_RATE) / (ms["merchant_txn_count"] + K)
        ms.drop(columns=["_fraud_sum"], inplace=True)
        _stats["merch_stats"]  = ms
        _stats["merch_global"] = GLOBAL_RATE

    ms = _stats["merch_stats"]
    df = df.merge(ms, on="merchant", how="left")
    gr = _stats.get("merch_global", 0)
    df["merchant_avg_amt"]   .fillna(df["amt"].mean(), inplace=True)
    df["merchant_std_amt"]   .fillna(df["amt"].std(),  inplace=True)
    df["merchant_fraud_rate"].fillna(gr,               inplace=True)
    df["merchant_txn_count"] .fillna(1,                inplace=True)
    df["amt_vs_merchant_avg"] = df["amt"] / (df["merchant_avg_amt"] + 1e-6)
    return df

In [ ]:
# 8. CATEGORY AGGREGATE FEATURES  (fit/transform pattern)
def add_category_features(df: pd.DataFrame, y: pd.Series = None, fit: bool = True) -> pd.DataFrame:
    """Category-level fraud rate with Bayesian smoothing. K=5."""
    df = df.copy()
    if fit:
        if y is None:
            raise ValueError("y required when fit=True")
        GLOBAL_RATE = y.mean()
        K = 5
        temp = df.copy(); temp["is_fraud"] = y.values
        cs = temp.groupby("category").agg(
            category_avg_amt   = ("amt",      "mean"),
            category_std_amt   = ("amt",      "std"),
            _count             = ("amt",      "count"),
            _fraud_sum         = ("is_fraud", "sum"),
        ).reset_index()
        cs["category_fraud_rate"] = (cs["_fraud_sum"] + K * GLOBAL_RATE) / (cs["_count"] + K)
        cs.drop(columns=["_fraud_sum", "_count"], inplace=True)
        _stats["cat_stats"]  = cs
        _stats["cat_global"] = GLOBAL_RATE

    cs = _stats["cat_stats"]
    df = df.merge(cs, on="category", how="left")
    gr = _stats.get("cat_global", 0)
    df["category_fraud_rate"].fillna(gr,               inplace=True)
    df["category_avg_amt"]   .fillna(df["amt"].mean(), inplace=True)
    df["category_std_amt"]   .fillna(df["amt"].std(),  inplace=True)
    df["amt_vs_category_avg"] = df["amt"] / (df["category_avg_amt"] + 1e-6)
    df["amt_category_zscore"] = (df["amt"] - df["category_avg_amt"]) / (df["category_std_amt"] + 1e-6)
    return df

In [ ]:
# 9. STATE FEATURES  (fit/transform pattern)
def add_state_features(df: pd.DataFrame, y: pd.Series = None, fit: bool = True) -> pd.DataFrame:
    """State-level fraud rate with Bayesian smoothing. K=20 (51 states, high txn counts)."""
    df = df.copy()
    if fit:
        if y is None:
            raise ValueError("y required when fit=True")
        GLOBAL_RATE = y.mean()
        K = 20
        temp = df.copy(); temp["is_fraud"] = y.values
        ss = temp.groupby("state").agg(
            state_avg_amt = ("amt",      "mean"),
            _count        = ("amt",      "count"),
            _fraud_sum    = ("is_fraud", "sum"),
        ).reset_index()
        ss["state_fraud_rate"] = (ss["_fraud_sum"] + K * GLOBAL_RATE) / (ss["_count"] + K)
        ss.drop(columns=["_fraud_sum", "_count"], inplace=True)
        _stats["state_stats"]  = ss
        _stats["state_global"] = GLOBAL_RATE

    ss = _stats["state_stats"]
    df = df.merge(ss, on="state", how="left")
    gr = _stats.get("state_global", 0)
    df["state_fraud_rate"].fillna(gr,               inplace=True)
    df["state_avg_amt"]   .fillna(df["amt"].mean(), inplace=True)
    return df

In [ ]:
# 10. JOB FEATURES
JOB_CATEGORIES = {
    "healthcare"  : ["nurse", "doctor", "physician", "therapist", "paramedic", "surgeon",
                     "pharmacist", "psychologist", "psychiatrist", "radiographer", "pathologist",
                     "biochemist", "physiotherapist", "optician", "paediatric"],
    "engineering" : ["engineer", "developer", "programmer", "scientist", "geologist",
                     "architect", "analyst", "technician", "physicist"],
    "finance"     : ["accountant", "banker", "adviser", "financial", "tax", "economist",
                     "investment", "actuary", "trader"],
    "legal"       : ["lawyer", "attorney", "solicitor", "judge", "paralegal", "probation"],
    "education"   : ["teacher", "professor", "lecturer", "researcher", "academic",
                     "educational", "librarian", "curator"],
    "creative"    : ["designer", "artist", "editor", "writer", "journalist", "photographer",
                     "film", "video", "music", "animator"],
    "transport"   : ["driver", "planner", "forwarder", "logistics", "transport", "naval"],
    "retail"      : ["manager", "officer", "consultant", "sales", "dealer", "buyer"],
    "trades"      : ["carpenter", "electrician", "plumber", "mechanic", "construction",
                     "surveyor", "arborist", "horticulturist"],
}

def _map_job(job_str: str) -> str:
    job_lower = str(job_str).lower()
    for category, keywords in JOB_CATEGORIES.items():
        if any(kw in job_lower for kw in keywords):
            return category
    return "other"

def add_job_features(df: pd.DataFrame, y: pd.Series = None, fit: bool = True) -> pd.DataFrame:
    """Map job to broad category and target-encode."""
    df = df.copy()

    df["job_category"] = df["job"].apply(_map_job)

    if fit:
        if y is None:
            raise ValueError("y (is_fraud) is required when fit=True.")

        temp = df.copy()
        temp["is_fraud"] = y.values

        GLOBAL_RATE = y.mean()
        K = 5
        job_stats = temp.groupby("job_category").agg(
            _count     = ("is_fraud", "count"),
            _fraud_sum = ("is_fraud", "sum"),
        ).reset_index()
        job_stats["job_fraud_rate"] = (job_stats["_fraud_sum"] + K * GLOBAL_RATE) / (job_stats["_count"] + K)
        job_stats.drop(columns=["_fraud_sum", "_count"], inplace=True)
        _stats["job_stats"]  = job_stats
        _stats["job_global"] = GLOBAL_RATE

    job_stats = _stats["job_stats"]
    df = df.merge(job_stats, on="job_category", how="left")
    df["job_fraud_rate"].fillna(_stats.get("job_global", 0), inplace=True)

    return df

In [ ]:
# 11. INTERACTION FEATURES
def add_interaction_features(df: pd.DataFrame) -> pd.DataFrame:
    """Cross-feature interactions that capture compound risk signals."""
    df = df.copy()

    # High amount + night time = elevated risk
    df["high_amt_night"]          = df["is_high_value"] * df["is_night"]

    # Far transaction + new merchant = strong fraud signal
    df["far_and_new_merchant"]    = df["is_far_transaction"] * df["is_new_merchant"]

    # High velocity + high amount
    df["rapid_high_amt"]          = (df["txn_count_last_1h"] > 2).astype(int) * df["is_high_value"]

    # Amount anomalous for both card AND category
    df["double_amt_anomaly"]      = (
        (abs(df["amt_zscore_card"]) > 2) & (abs(df["amt_category_zscore"]) > 2)
    ).astype(int)

    # Weekend + online category (shopping_net, misc_net, grocery_net)
    online_cats = ["shopping_net", "misc_net", "grocery_net"]
    df["weekend_online"]          = (
        df["is_weekend"] & df["category"].isin(online_cats)
    ).astype(int)

    # Distance × merchant fraud rate (risky location + risky merchant)
    # Use log_distance — distance_from_home dropped as raw version
    df["dist_x_merch_fraud"]      = df["log_distance"] * df["merchant_fraud_rate"]

    # Senior + high value
    df["senior_high_value"]       = df["is_senior"] * df["is_high_value"]

    return df

In [ ]:
# 12. "MAGIC" — UID CONSTRUCTION + UID AGGREGATIONS
def build_uid(df: pd.DataFrame) -> pd.DataFrame:
    """
    Construct a Unique Client ID (UID) from stable cardholder attributes.
    UID = cc_num + "_" + zip + "_" + dob  (string concatenation)
    The UID is a GROUP KEY — dropped after aggregation, never used as a model feature.
    """
    df = df.copy()
    df["uid"] = (
        df["cc_num"].astype(str) + "_" +
        df["zip"].astype(str)    + "_" +
        df["dob"].astype(str)
    )
    return df


def add_uid_frequency_encoding(df: pd.DataFrame, fit: bool = True) -> pd.DataFrame:
    """Frequency encode the UID — how many transactions does this client have?"""
    df = df.copy()
    if fit:
        _stats["uid_counts"] = df["uid"].value_counts().to_dict()
    uid_counts     = _stats["uid_counts"]
    df["uid_freq"]     = df["uid"].map(uid_counts).fillna(1).astype(int)
    df["log_uid_freq"] = np.log1p(df["uid_freq"])
    return df


def add_uid_aggregations(df: pd.DataFrame, y: pd.Series = None, fit: bool = True) -> pd.DataFrame:
    """
    Core Magic step from cdeotte.
    Bayesian-smoothed uid_fraud_rate separates new UIDs (unknown risk) from
    established clean UIDs — raw mean collapses both to 0. K=10.
    uid_is_new flags clients with <=3 transactions for explicit modelling.
    uid_dist_mean uses log_distance (distance_from_home is dropped).
    """
    df = df.copy()
    if fit:
        if y is None:
            raise ValueError("y (is_fraud) is required when fit=True for UID fraud rate.")
        GLOBAL_RATE = y.mean()
        K = 10
        temp = df.copy(); temp["is_fraud"] = y.values
        uid_agg = temp.groupby("uid").agg(
            uid_amt_mean   = ("amt",      "mean"),
            uid_amt_std    = ("amt",      "std"),
            uid_amt_max    = ("amt",      "max"),
            uid_amt_min    = ("amt",      "min"),
            uid_hour_mean  = ("hour",     "mean"),
            uid_txn_total  = ("amt",      "count"),
            _fraud_sum     = ("is_fraud", "sum"),
        ).reset_index()
        uid_agg["uid_fraud_rate"] = (uid_agg["_fraud_sum"] + K * GLOBAL_RATE) / (uid_agg["uid_txn_total"] + K)
        uid_agg["uid_is_new"]     = (uid_agg["uid_txn_total"] <= 3).astype(np.int8)
        uid_agg.drop(columns=["_fraud_sum"], inplace=True)
        dist_col = "log_distance" if "log_distance" in temp.columns else "distance_from_home"
        if dist_col in temp.columns:
            uid_dist = temp.groupby("uid")[dist_col].mean().reset_index()
            uid_dist.columns = ["uid", "uid_dist_mean"]
            uid_agg = uid_agg.merge(uid_dist, on="uid", how="left")
        else:
            uid_agg["uid_dist_mean"] = np.nan
        _stats["uid_agg"]    = uid_agg
        _stats["uid_global"] = GLOBAL_RATE

    uid_agg = _stats["uid_agg"]
    df = df.merge(uid_agg, on="uid", how="left")
    gr = _stats.get("uid_global", 0)
    df["uid_amt_mean"]  .fillna(df["amt"].mean(), inplace=True)
    df["uid_amt_std"]   .fillna(df["amt"].std(),  inplace=True)
    df["uid_fraud_rate"].fillna(gr,               inplace=True)
    df["uid_txn_total"] .fillna(1,                inplace=True)
    df["uid_dist_mean"] .fillna(0,                inplace=True)
    df["uid_is_new"]     = df["uid_is_new"].fillna(1).astype(np.int8)
    df["amt_vs_uid_mean"]   = df["amt"] / (df["uid_amt_mean"] + 1e-6)
    df["amt_uid_zscore"]    = (df["amt"] - df["uid_amt_mean"]) / (df["uid_amt_std"] + 1e-6)
    df["is_above_uid_max"]  = (df["amt"] > df["uid_amt_max"]).astype(int)
    df["uid_hour_deviation"]= abs(df["hour"] - df["uid_hour_mean"])
    return df


def add_uid_velocity(df: pd.DataFrame) -> pd.DataFrame:
    """Per-UID velocity features."""
    df = df.copy()
    df["uid_time_since_last"]     = df.groupby("uid")["unix_time"].diff().fillna(999999)
    df["log_uid_time_since_last"] = np.log1p(df["uid_time_since_last"])
    df["uid_cumulative_txns"]     = df.groupby("uid").cumcount()
    return df


def postprocess_predictions(pred_proba: pd.Series, uid_series: pd.Series) -> pd.Series:
    """cdeotte post-processing: replace per-UID predictions with UID mean probability."""
    df_tmp   = pd.DataFrame({"pred": pred_proba.values, "uid": uid_series.values})
    uid_mean = df_tmp.groupby("uid")["pred"].transform("mean")
    return pd.Series(uid_mean.values, index=pred_proba.index)

In [ ]:
# 13. CLEANUP — drop raw columns no longer needed
COLUMNS_TO_DROP = [
    "trans_date_trans_time",
    "trans_datetime",
    "dob",
    "dob_parsed",
    "first",
    "last",
    "street",
    "trans_num",
    "unix_time",            # used for velocity, now redundant
    "city",                 # raw demographic — 879 unique values, not a fraud signal
    "zip",                  # raw identifier — geo signal covered by distance features
    "merchant",             # raw string — use merchant_fraud_rate, merchant_avg_amt instead
    "job",                  # raw string — use job_fraud_rate, job_category instead
    "state",                # raw string — use state_fraud_rate instead
    "category",             # raw string — use category_fraud_rate instead
    "uid_freq",             # redundant with uid_cumulative_txns (same signal, less stable)
    "log_uid_freq",         # same redundancy as uid_freq
    "distance_from_home",   # prefer log_distance
    "time_since_last_txn",  # prefer log_time_since_last
    "uid_time_since_last",  # prefer log_uid_time_since_last
]

def drop_raw_columns(df: pd.DataFrame, also_drop_cc_num: bool = True) -> pd.DataFrame:
    df = df.copy()
    cols = COLUMNS_TO_DROP.copy()
    if also_drop_cc_num:
        cols.append("cc_num")
    return df.drop(columns=[c for c in cols if c in df.columns])

In [ ]:
# 14. FINAL FUNCTION
def engineer_all_features(
    X: pd.DataFrame,
    y: pd.Series = None,
    fit: bool = True,
    drop_cc_num: bool = True,
    verbose: bool = True
) -> pd.DataFrame:
    """
    Full feature engineering pipeline.

    Parameters
    ----------
    X          : Raw feature DataFrame (is_fraud already removed)
    y          : is_fraud Series — required when fit=True for target encoding
    fit        : True for training data, False for validation/test data
    drop_cc_num: Whether to drop cc_num in final output (default True)
    verbose    : Print progress steps

    Returns
    -------
    pd.DataFrame with all engineered features
    """
    if fit and y is None:
        raise ValueError("y must be provided when fit=True (needed for target encoding).")

    def log(msg):
        if verbose:
            print(f"  [feature_eng] {msg}")

    log("Step 0 — Preprocessing (parse dates, sort by card+time)...")
    df = preprocess(X)

    log("Step 1 — Temporal features...")
    df = add_temporal_features(df)

    log("Step 2 — Age features...")
    df = add_age_features(df)

    log("Step 3 — Geographic / distance features...")
    df = add_geographic_features(df)

    log("Step 4 — Amount features...")
    df = add_amount_features(df)

    log("Step 5 — Velocity / behavioral features (slowest step)...")
    df = add_velocity_features(df)

    log("Step 6 — Cardholder profile aggregates...")
    df = add_card_profile_features(df, fit=fit)

    log("Step 7 — Merchant aggregate features...")
    df = add_merchant_features(df, y=y, fit=fit)

    log("Step 8 — Category aggregate features...")
    df = add_category_features(df, y=y, fit=fit)

    log("Step 9 — State fraud rate encoding...")
    df = add_state_features(df, y=y, fit=fit)

    log("Step 10 — Job features...")
    df = add_job_features(df, y=y, fit=fit)

    log("Step 11 — Interaction features...")
    df = add_interaction_features(df)

    # ── cdeotte "Magic" steps ─────────────────────────────────
    log("Step 12 — [MAGIC] Build UID (stable client identifier)...")
    df = build_uid(df)

    log("Step 13 — [MAGIC] UID frequency encoding...")
    df = add_uid_frequency_encoding(df, fit=fit)

    log("Step 14 — [MAGIC] UID aggregations (amt mean/std, fraud rate per client)...")
    df = add_uid_aggregations(df, y=y, fit=fit)

    log("Step 15 — [MAGIC] UID velocity features...")
    df = add_uid_velocity(df)

    # Drop UID after aggregations are done — it's a groupby key, not a feature
    # (high cardinality string would cause overfitting if left in)
    df.drop(columns=["uid"], inplace=True)
    # ─────────────────────────────────────────────────────────

    log("Step 16 — Dropping raw columns...")
    df = drop_raw_columns(df, also_drop_cc_num=drop_cc_num)

    log(f"Done. Output shape: {df.shape}")
    return df



# MODEL-SPECIFIC SPLITS



---


After engineer_all_features() produces a unified engineered DataFrame,
each model needs different treatment of categorical columns:

   XGBoost  — cannot handle strings at all. All categoricals must be
               ordinal/label encoded into integers. Cyclic features
               (hour_sin/cos) are valuable since XGB splits on them well.

   LightGBM — supports native categoricals via pd.Categorical dtype,
               but still benefits from explicit dtype marking. Cyclic
               features still useful. High-cardinality cats like merchant
               should be target-encoded rather than passed raw.

   CatBoost — has its own internal ordered target encoding. Pass raw
               string categoricals directly; CatBoost does better with
               them than any manual encoding would. Do NOT target-encode
               before passing to CatBoost — it will double-encode.
               Remove cyclic sin/cos features; CatBoost handles raw
               hour/dow integers natively and more robustly.

 Each split function takes the output of engineer_all_features() and
 returns a model-ready DataFrame. They are pure transformations — no
 fitting required (all stats were captured during feature engineering).


In [ ]:
# Columns that are categorical strings in the engineered DataFrame
# city, merchant, job, state, category all removed — see COLUMNS_TO_DROP
_CAT_COLS = [
    "gender",
    "job_category",      # bucketed job — generalizes to unseen job titles
    "age_group",
    "amt_bin",
    "card_home_state",
    "card_fav_cat",
]

# Cyclic features — useful for XGB/LGB, redundant for CatBoost
_CYCLIC_COLS = ["hour_sin", "hour_cos", "dow_sin", "dow_cos"]

_RAW_TIME_COLS = ["hour", "day_of_week"]

In [ ]:
def prepare_for_xgboost(df: pd.DataFrame) -> pd.DataFrame:
    """
    Prepare engineered DataFrame for XGBoost.

    XGBoost requirements:
      - No string columns whatsoever (will raise an error)
      - All categoricals label-encoded to integers
      - NaN is fine — XGBoost handles missing values internally
      - Cyclic features (hour_sin/cos) kept — XGB splits on them well
      - Boolean flags already int — no change needed

    Encoding strategy:
      - Low-cardinality cats (gender, age_group, amt_bin): label encode
      - High-cardinality cats (merchant, category, state, job, job_category,
        card_home_state, card_fav_cat): label encode via pd.factorize
        Note: target-encoded versions (merchant_fraud_rate, category_fraud_rate,
        state_fraud_rate, job_fraud_rate) already exist as floats from Step 7-10,
        so raw string cols can be label-encoded or dropped. We keep both to give
        XGB the choice — it will deprioritise redundant ones automatically.
    """
    df = df.copy()

    for col in _CAT_COLS:
        if col in df.columns:
            # pd.factorize: assigns integer codes, -1 for NaN → becomes -1 (XGB handles)
            codes, _ = pd.factorize(df[col])
            df[col] = codes

    # Ensure no remaining object columns (safety check)
    obj_cols = df.select_dtypes(include="object").columns.tolist()
    if obj_cols:
        for col in obj_cols:
            codes, _ = pd.factorize(df[col])
            df[col] = codes

    return df

In [ ]:
def prepare_for_lightgbm(df: pd.DataFrame) -> pd.DataFrame:
    """
    Prepare engineered DataFrame for LightGBM.

    LightGBM requirements:
      - Categorical columns should be cast to pd.Categorical dtype
        (LightGBM detects them and uses its own optimal binning)
      - Alternatively pass as integers and specify categorical_feature=
        in the fit() call — both work; dtype approach is more portable
      - NaN is fine — LightGBM handles natively
      - Cyclic features kept
      - High-cardinality strings (merchant: 693 classes) are fine for LGB
        since it uses histogram-based splits and handles high cardinality
        better than XGB. Mark as categorical rather than label encoding.

    Note: target-encoded float versions (merchant_fraud_rate etc.) coexist
    alongside pd.Categorical columns. LightGBM will use whichever adds more
    split gain — both are useful and non-redundant from LGB's perspective.
    """
    df = df.copy()

    for col in _CAT_COLS:
        if col in df.columns:
            # Fill NaN with "unknown" before casting to Categorical
            df[col] = df[col].fillna("unknown").astype("category")

    return df


In [ ]:
def prepare_for_catboost(df: pd.DataFrame) -> pd.DataFrame:
    """
    Prepare engineered DataFrame for CatBoost.

    CatBoost requirements:
      - String categoricals passed AS-IS (raw strings) — CatBoost applies
        its own ordered target statistics internally, which is generally
        superior to any pre-computed target encoding
      - IMPORTANT: remove pre-computed target-encoded float versions of
        categorical columns (merchant_fraud_rate, category_fraud_rate,
        state_fraud_rate, job_fraud_rate) — if both the raw categorical
        AND the target-encoded float are present, CatBoost will use the
        float and ignore the categorical, wasting its internal encoder.
        The raw categorical gives CatBoost MORE information.
      - Remove cyclic sin/cos features — CatBoost handles raw hour/dow
        integers natively via its symmetric tree structure. Sin/cos can
        actually confuse its splitter slightly.
      - NaN in categoricals: fill with "nan" string (CatBoost handles
        "nan" as its own category, which is better than truly missing)
      - NaN in numerics: CatBoost handles natively, no fill needed
      - Return cat_features index list alongside DataFrame so caller
        can pass it directly to CatBoostClassifier(cat_features=...)

    Returns
    -------
    tuple: (df, cat_features_indices)
        df               : model-ready DataFrame
        cat_features     : list of column NAMES that are categorical
                           (pass to CatBoost as cat_features=cat_features)
    """
    df = df.copy()

    # Remove target-encoded float versions — CatBoost will re-encode internally
    _te_to_remove = [
        "merchant_fraud_rate",
        "category_fraud_rate",
        "state_fraud_rate",
        "job_fraud_rate",
        "uid_fraud_rate",   # CatBoost will derive this from uid aggregations
    ]
    df.drop(columns=[c for c in _te_to_remove if c in df.columns], inplace=True)

    # Remove cyclic features — raw hour/dow are better for CatBoost's splitter
    df.drop(columns=[c for c in _CYCLIC_COLS if c in df.columns], inplace=True)

    # Fill NaN in categorical string columns with "nan" string
    for col in _CAT_COLS:
        if col in df.columns:
            df[col] = df[col].fillna("nan").astype(str)

    # Build list of categorical column names present in df
    cat_features = [col for col in _CAT_COLS if col in df.columns]

    return df, cat_features

In [ ]:
def split_for_models(
    X_eng: pd.DataFrame,
    verbose: bool = True
) -> dict:
    """
    Split a single engineered DataFrame into three model-ready DataFrames.

    Parameters
    ----------
    X_eng   : Output of engineer_all_features()
    verbose : Print shape summary for each split

    Returns
    -------
    dict with keys:
        "xgb"              : DataFrame ready for XGBoost
        "lgb"              : DataFrame ready for LightGBM
        "cgb"              : DataFrame ready for CatBoost
        "cgb_cat_features" : list of categorical column names for CatBoost

    Usage
    -----
        splits = split_for_models(X_train_eng)
        X_xgb = splits["xgb"]
        X_lgb = splits["lgb"]
        X_cgb, cat_features = splits["cgb"], splits["cgb_cat_features"]

        # CatBoost
        model_cgb.fit(X_cgb, y_train, cat_features=cat_features)
    """
    def log(msg):
        if verbose:
            print(f"  [model_split] {msg}")

    log("Preparing X_xgb — label-encoding all categoricals for XGBoost...")
    X_xgb = prepare_for_xgboost(X_eng)

    log("Preparing X_lgb — casting categoricals to pd.Categorical for LightGBM...")
    X_lgb = prepare_for_lightgbm(X_eng)

    log("Preparing X_cgb — passing raw strings, removing pre-encoded targets for CatBoost...")
    X_cgb, cat_features = prepare_for_catboost(X_eng)

    if verbose:
        print(f"\n  ┌─────────────┬────────────┬──────────────────────────────┐")
        print(f"  │ Split       │ Shape      │ Notes                        │")
        print(f"  ├─────────────┼────────────┼──────────────────────────────┤")
        print(f"  │ X_xgb       │ {str(X_xgb.shape):<10} │ All ints/floats, label-enc   │")
        print(f"  │ X_lgb       │ {str(X_lgb.shape):<10} │ Includes pd.Categorical cols │")
        print(f"  │ X_cgb       │ {str(X_cgb.shape):<10} │ Raw strings + no TE floats   │")
        print(f"  └─────────────┴────────────┴──────────────────────────────┘")
        print(f"\n  CatBoost cat_features ({len(cat_features)}): {cat_features}")

    return {
        "xgb"             : X_xgb,
        "lgb"             : X_lgb,
        "cgb"             : X_cgb,
        "cgb_cat_features": cat_features,
    }

#Final Function

In [ ]:
#FINAL FULL FEATURE ENGINEERING WRAPPER FUNCTION
def full_pipeline(
    X: pd.DataFrame,
    y: pd.Series = None,
    fit: bool = True,
    verbose: bool = True
) -> dict:
    """
    One-call wrapper: runs engineer_all_features() then split_for_models().

    Returns the same dict as split_for_models() — use this for the cleanest
    end-to-end flow.

    Usage
    -----
        # Training
        train_splits = full_pipeline(X_train, y=y_train, fit=True)
        X_xgb = train_splits["xgb"]
        X_lgb = train_splits["lgb"]
        X_cgb = train_splits["cgb"]
        cat_features = train_splits["cgb_cat_features"]

        # Test/Val (uses stored stats from training — no leakage)
        test_splits = full_pipeline(X_test, y=None, fit=False)
    """
    X_eng = engineer_all_features(X, y=y, fit=fit, verbose=verbose)
    return split_for_models(X_eng, verbose=verbose)

#Main

In [ ]:
# Assume X and y are your DataFrames from loading the dataset
y = df['is_fraud']
X = df.drop(columns=['is_fraud'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ── Feature engineering ───────────────────────────────────────
# Fit on training data (computes and stores all aggregation stats)
X_train_eng = engineer_all_features(X_train, y=y_train, fit=True)

# Transform test data (uses stored stats — no leakage)
X_test_eng  = engineer_all_features(X_test,  y=None,   fit=False)

print(f"Training features: {X_train_eng.shape[1]}")
print(f"Feature names:     {X_train_eng.columns.tolist()}")

# ── Postprocessing (cdeotte magic step 3) ────────────────────
# After training your model and getting raw probabilities:

# Build UID for test set BEFORE dropping it (keep separately)
X_test_with_uid = X_test.copy()
X_test_with_uid = preprocess(X_test_with_uid)
X_test_with_uid = build_uid(X_test_with_uid)   # uid column available here

# Placeholder for model.predict_proba. This line will cause an error without a trained model.
# raw_preds    = pd.Series(model.predict_proba(X_test_eng)[:, 1])
# smooth_preds = postprocess_predictions(raw_preds, X_test_with_uid["uid"])

# smooth_preds will have slightly higher AUC than raw_preds
# because all transactions from the same fraudulent client
# get pulled toward a consistent fraud probability.


In [ ]:
train_splits = full_pipeline(X_train, y=y_train, fit=True)
test_splits  = full_pipeline(X_test,  y=None,    fit=False)

X_xgb,  X_xgb_test  = train_splits["xgb"],  test_splits["xgb"]
X_lgb,  X_lgb_test  = train_splits["lgb"],  test_splits["lgb"]
X_cgb,  X_cgb_test  = train_splits["cgb"],  test_splits["cgb"]
cat_features         = train_splits["cgb_cat_features"]

In [ ]:
X_xgb

In [ ]:
X_xgb_test

#SHAP

In [ ]:
import gc
from sklearn.model_selection import StratifiedKFold

# ──────────────────────────────────────────────────────────────
# FEATURE GROUPS
# ──────────────────────────────────────────────────────────────
FEATURE_GROUPS = {
    "Velocity": [
        "log_time_since_last",
        "txn_count_last_1h", "txn_count_last_24h", "txn_count_last_7d",
        "amt_last_txn", "amt_diff_from_last",
        "unique_merchants_24h", "unique_categories_24h",
        "is_new_merchant", "is_new_category",
        "log_uid_time_since_last",
        "uid_cumulative_txns",
    ],
    "Amount Statistics": [
        "amt", "log_amt", "amt_round", "amt_cents_only", "amt_bin", "is_high_value",
        "amt_mean_7d", "amt_std_7d", "amt_zscore_card", "amt_vs_mean_7d",
        "amt_last_txn", "amt_diff_from_last",
        "amt_vs_merchant_avg", "amt_vs_category_avg",
        "amt_category_zscore", "card_amt_zscore", "is_above_card_max",
    ],
    "Time & Periodicity": [
        "hour", "day_of_week", "day_of_month", "month", "quarter",
        "is_weekend", "is_night", "is_business_hours",
        "hour_sin", "hour_cos", "dow_sin", "dow_cos",
        "hour_deviation_from_norm", "uid_hour_mean", "uid_hour_deviation",
    ],
    "Geography": [
        "log_distance", "is_far_transaction",
        "merch_lat_diff", "merch_long_diff",
        "lat", "long", "merch_lat", "merch_long",
        "city_pop", "log_city_pop",
    ],
    "User Profile & Card History": [
        "card_avg_amt", "card_std_amt", "card_max_amt", "card_txn_count",
        "card_home_state", "card_fav_cat", "card_avg_hour",
        "is_unusual_state", "is_unusual_category",
    ],
    "UID Magic (cdeotte)": [
        "uid_amt_mean", "uid_amt_std", "uid_amt_max", "uid_amt_min",
        "uid_fraud_rate", "uid_txn_total", "uid_dist_mean",
        "amt_vs_uid_mean", "amt_uid_zscore", "is_above_uid_max",
        "uid_is_new",
    ],
    "Merchant Signals": [
        "merchant_avg_amt", "merchant_std_amt",
        "merchant_txn_count", "merchant_fraud_rate",
    ],
    "Category Signals": [
        "category_avg_amt", "category_std_amt", "category_fraud_rate",
    ],
    "Demographics": [
        "age", "age_group", "is_senior", "gender",
        "job_category", "job_fraud_rate",
    ],
    "State Signals": [
        "state_fraud_rate", "state_avg_amt",
    ],
    "Interactions": [
        "high_amt_night", "far_and_new_merchant",
        "rapid_high_amt", "double_amt_anomaly",
        "weekend_online", "dist_x_merch_fraud", "senior_high_value",
    ],
}

FEATURE_TO_GROUP = {feat: grp for grp, feats in FEATURE_GROUPS.items() for feat in feats}


# ──────────────────────────────────────────────────────────────
# STEP 1 — LIGHT LIGHTGBM TRAINING
# ──────────────────────────────────────────────────────────────
print("=" * 65)
print("STEP 1 — Light LightGBM for SHAP")
print("=" * 65)

_neg = (y_train == 0).sum()
_pos = (y_train == 1).sum()
SPW  = _neg / _pos

shap_lgb_params = dict(
    n_estimators     = 300,
    learning_rate    = 0.05,
    max_depth        = 6,
    num_leaves       = 63,
    min_child_samples= 20,
    subsample        = 0.8,
    subsample_freq   = 1,
    colsample_bytree = 0.8,
    reg_alpha        = 0.1,
    reg_lambda       = 1.0,
    scale_pos_weight = SPW,
    objective        = "binary",
    random_state     = 42,
    n_jobs           = -1,
    verbose          = -1,
)

lgb_cat_cols = X_lgb.select_dtypes(include="category").columns.tolist()

X_lgb_tr, X_lgb_val, y_lgb_tr, y_lgb_val = train_test_split(
    X_lgb, y_train, test_size=0.2, stratify=y_train, random_state=42
)

shap_model = LGBMClassifier(**shap_lgb_params)
shap_model.fit(X_lgb_tr, y_lgb_tr, categorical_feature=lgb_cat_cols)

val_auc = roc_auc_score(y_lgb_val, shap_model.predict_proba(X_lgb_val)[:, 1])
val_pr  = average_precision_score(y_lgb_val, shap_model.predict_proba(X_lgb_val)[:, 1])
print(f"\n  Light model — Val AUC-ROC: {val_auc:.4f}  |  Val AUC-PR: {val_pr:.4f}")
print(f"  (This is for SHAP only — not the production model)\n")

In [ ]:
# ──────────────────────────────────────────────────────────────
# STEP 2A — GLOBAL SHAP IMPORTANCE ON VALIDATION SET
# ──────────────────────────────────────────────────────────────
print("=" * 65)
print("STEP 2A — Global SHAP Importance (mean |SHAP|)")
print("=" * 65)

SHAP_SAMPLE   = min(5000, len(X_lgb_val))
X_shap_sample = X_lgb_val.sample(n=SHAP_SAMPLE, random_state=42)

explainer   = shap.TreeExplainer(shap_model)
shap_values = explainer.shap_values(X_shap_sample)
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

mean_shap     = np.abs(sv).mean(axis=0)
feature_names = X_lgb_val.columns.tolist()

shap_df = pd.DataFrame({
    "feature"  : feature_names,
    "mean_shap": mean_shap,
    "group"    : [FEATURE_TO_GROUP.get(f, "Other") for f in feature_names],
}).sort_values("mean_shap", ascending=False).reset_index(drop=True)

shap_df["rank"]         = shap_df.index + 1
shap_df["shap_pct"]     = shap_df["mean_shap"] / shap_df["mean_shap"].sum() * 100
shap_df["shap_cum_pct"] = shap_df["shap_pct"].cumsum()

print(f"\n  Top 30 features by mean |SHAP|:\n")
print(shap_df[["rank", "feature", "group", "mean_shap", "shap_pct", "shap_cum_pct"]]
      .head(30).to_string(index=False))

In [ ]:
# ──────────────────────────────────────────────────────────────
# STEP 2B — FOLD STABILITY (SHAP rank variance across 5 folds)
# ──────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("STEP 2B — SHAP Rank Stability Across 5 Folds")
print("=" * 65)

SKF           = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
n_features    = len(feature_names)
HIGH_STD_RANK = int(0.25 * n_features)
print(f"  n_features    = {n_features}")
print(f"  HIGH_STD_RANK = {HIGH_STD_RANK}  (= 0.25 × {n_features})")

fold_shap_ranks = []
for fold_idx, (tr_idx, val_idx) in enumerate(SKF.split(X_lgb, y_train)):
    X_fold_tr  = X_lgb.iloc[tr_idx]
    y_fold_tr  = y_train.iloc[tr_idx]
    X_fold_val = X_lgb.iloc[val_idx]
    fold_model = LGBMClassifier(**shap_lgb_params)
    fold_model.fit(X_fold_tr, y_fold_tr, categorical_feature=lgb_cat_cols)
    fold_explainer = shap.TreeExplainer(fold_model)   # fresh per fold — never reuse global explainer
    _sample   = X_fold_val.sample(n=min(2000, len(X_fold_val)), random_state=42)
    _sv_fold  = fold_explainer.shap_values(_sample)
    _sv_fold  = _sv_fold[1] if isinstance(_sv_fold, list) else _sv_fold
    fold_mean_shap = np.abs(_sv_fold).mean(axis=0)
    fold_ranks = pd.Series(fold_mean_shap, index=feature_names).rank(ascending=False)
    fold_shap_ranks.append(fold_ranks)
    print(f"  Fold {fold_idx+1}/5 done — top feature: {feature_names[fold_mean_shap.argmax()]}")
    del fold_model, fold_explainer, _sv_fold, _sample; gc.collect()

stability_df = pd.DataFrame(fold_shap_ranks).T
stability_df.columns = [f"fold_{i+1}_rank" for i in range(5)]
stability_df["mean_rank"] = stability_df.mean(axis=1)
stability_df["std_rank"]  = stability_df.std(axis=1)
stability_df["feature"]   = feature_names
stability_df = stability_df.sort_values("mean_rank").reset_index(drop=True)

shap_df = shap_df.merge(
    stability_df[["feature", "mean_rank", "std_rank"]], on="feature", how="left"
)

print(f"\n  Top 30 features by stability (mean_rank ± std_rank):\n")
print(stability_df[["feature", "mean_rank", "std_rank"]
      + [f"fold_{i+1}_rank" for i in range(5)]]
      .head(30).to_string(index=False))

In [ ]:
# ──────────────────────────────────────────────────────────────
# STEP 3 — GROUP-LEVEL IMPORTANCE
# ──────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("STEP 3 — Group-Level SHAP Importance")
print("=" * 65)

group_importance = (
    shap_df.groupby("group")["mean_shap"]
    .sum().sort_values(ascending=False).reset_index()
)
group_importance["pct"] = group_importance["mean_shap"] / group_importance["mean_shap"].sum() * 100

print(f"\n  {'Group':<30} {'Total SHAP':>12}  {'% of total':>10}")
print("  " + "─" * 56)
for _, row in group_importance.iterrows():
    bar = "█" * int(row["pct"] / 2)
    print(f"  {row['group']:<30} {row['mean_shap']:>12.4f}  {row['pct']:>9.1f}%  {bar}")

In [ ]:

# ──────────────────────────────────────────────────────────────
# STEP 4 — PRUNING LOGIC
# Mark each feature for KEEP / REMOVE with explicit reason
# ──────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("STEP 4 — Feature Pruning")
print("=" * 65)

# ── 4a. Correlation check ─────────────────────────────────────
# Compute correlation on numeric columns only
# Flag pairs with |r| > 0.90 where the weaker feature is a removal candidate
print("\n  Computing pairwise correlations...")
num_cols  = X_lgb_val.select_dtypes(include=[np.number]).columns.tolist()
corr_mat  = X_lgb_val[num_cols].corr().abs()

# Upper triangle only — avoid duplicate pairs
upper     = corr_mat.where(np.triu(np.ones(corr_mat.shape), k=1).astype(bool))
high_corr = (
    upper.stack()
    .reset_index()
    .rename(columns={"level_0": "feat_a", "level_1": "feat_b", 0: "corr"})
    .query("corr > 0.90")
    .sort_values("corr", ascending=False)
)

# For each high-corr pair, mark the WEAKER feature (higher mean_rank = weaker)
corr_removal_candidates = set()
for _, row in high_corr.iterrows():
    rank_a = shap_df.set_index("feature")["mean_rank"].get(row["feat_a"], 9999)
    rank_b = shap_df.set_index("feature")["mean_rank"].get(row["feat_b"], 9999)
    weaker = row["feat_b"] if rank_a <= rank_b else row["feat_a"]
    corr_removal_candidates.add(weaker)

print(f"  High-correlation pairs (|r|>0.90): {len(high_corr)}")
print(f"  Weaker correlated features flagged: {len(corr_removal_candidates)}")

# ── 4b. Pruning criteria ──────────────────────────────────────
NEAR_ZERO_SHAP_PCT = 0.05    # features contributing < 0.05% of total SHAP
# HIGH_STD_RANK already set above as int(0.25 * n_features) — Fix 3

shap_df["remove_low_shap"]    = shap_df["shap_pct"] < NEAR_ZERO_SHAP_PCT
shap_df["remove_correlated"]  = shap_df["feature"].isin(corr_removal_candidates)
shap_df["remove_unstable"]    = shap_df["std_rank"] > HIGH_STD_RANK

# ── 4c. Override: always KEEP features with distinct fraud mechanism ──
# Even if weak in SHAP — these represent separate fraud modalities
# and removing them would create a blind spot
ALWAYS_KEEP = {
    # Card testing signal — micro txn before big fraud
    "amt_cents_only",
    # Time-of-day fraud modality — night transactions
    "is_night", "is_business_hours",
    # Geography — card-not-present vs in-person divergence
    # Fix 4: use log_distance (raw distance_from_home dropped per log preference)
    "is_far_transaction", "log_distance",
    # New entity exposure — first time at this merchant / category
    "is_new_merchant", "is_new_category",
    # cdeotte magic — client-level fraud history is irreplaceable
    # Fix 5: removed duplicate "uid_fraud_rate" entry
    "uid_fraud_rate",
    # Senior targeting modality
    "is_senior",
    # Rapid multi-merchant velocity (card testing)
    "unique_merchants_24h", "txn_count_last_1h",
}

shap_df["always_keep"] = shap_df["feature"].isin(ALWAYS_KEEP)

# Final decision
def _pruning_decision(row):
    if row["always_keep"]:
        return "KEEP", "protected — distinct fraud mechanism"
    remove_flags = []
    if row["remove_low_shap"]:
        remove_flags.append("near-zero SHAP")
    if row["remove_correlated"]:
        remove_flags.append("correlated with stronger feature")
    if row["remove_unstable"]:
        remove_flags.append("unstable across folds")
    if len(remove_flags) >= 2:      # remove only if 2+ flags triggered
        return "REMOVE", " + ".join(remove_flags)
    elif len(remove_flags) == 1:
        return "REVIEW", remove_flags[0]   # one flag alone → manual review
    return "KEEP", "—"

shap_df[["decision", "reason"]] = shap_df.apply(
    _pruning_decision, axis=1, result_type="expand"
)

# ── Fix 6: Safety guards ──────────────────────────────────────
# Guard 1: all decisions must be valid labels
valid_decisions = {"KEEP", "REVIEW", "REMOVE"}
invalid = set(shap_df["decision"].unique()) - valid_decisions
assert not invalid, (
    f"Invalid pruning decision(s) found: {invalid}. "
    f"_pruning_decision() must return only: {valid_decisions}"
)

# Guard 2: minimum retained feature count to prevent recall collapse
# REVIEW features are conservatively treated as KEEP for this check
MIN_FEATURES = 60
safe_count = len(shap_df[shap_df["decision"].isin(["KEEP", "REVIEW"])])
assert safe_count >= MIN_FEATURES, (
    f"Pruning too aggressive: only {safe_count} features retained "
    f"(KEEP + REVIEW). Minimum required is {MIN_FEATURES} to preserve "
    f"multi-modal fraud coverage. Loosen NEAR_ZERO_SHAP_PCT or review "
    f"ALWAYS_KEEP set."
)

# ── 4d. Summary ───────────────────────────────────────────────
keep_features   = shap_df[shap_df["decision"] == "KEEP"]["feature"].tolist()
remove_features = shap_df[shap_df["decision"] == "REMOVE"]["feature"].tolist()
review_features = shap_df[shap_df["decision"] == "REVIEW"]["feature"].tolist()

print(f"\n  Pruning Results:")
print(f"  {'─'*40}")
print(f"  KEEP   : {len(keep_features):>3} features")
print(f"  REVIEW : {len(review_features):>3} features  ← inspect manually")
print(f"  REMOVE : {len(remove_features):>3} features")
print(f"  {'─'*40}")
print(f"  Total  : {len(shap_df):>3} features")

print(f"\n  Features marked REMOVE:\n")
for f in remove_features:
    row = shap_df[shap_df["feature"] == f].iloc[0]
    print(f"    {f:<45} [{row['reason']}]")

print(f"\n  Features marked REVIEW (check manually):\n")
for f in review_features:
    row = shap_df[shap_df["feature"] == f].iloc[0]
    print(f"    {f:<45} [{row['reason']}]")

# ── 4e. Target feature count check ───────────────────────────
target_low, target_high = 70, 90
final_count = len(keep_features)
print(f"\n  Target range: {target_low}–{target_high} features")
print(f"  Current KEEP count: {final_count}")

if final_count > target_high:
    print(f"  ⚠  {final_count - target_high} over target — "
          f"consider moving some REVIEW→REMOVE or tightening NEAR_ZERO_SHAP_PCT")
elif final_count < target_low:
    print(f"  ⚠  {target_low - final_count} under target — "
          f"consider moving some REVIEW→KEEP")
else:
    print(f"  ✓  Within target range")


# ──────────────────────────────────────────────────────────────
# STEP 5 — VISUALISATIONS
# ──────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("STEP 5 — SHAP Visualisations")
print("=" * 65)

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle("SHAP Feature Analysis — LightGBM (Light Model)", fontsize=14, y=1.01)

# ── Plot 1: Top 30 features by mean |SHAP| ────────────────────
ax1 = axes[0, 0]
top30 = shap_df.head(30).sort_values("mean_shap")
colors = top30["group"].map({
    g: c for g, c in zip(
        shap_df["group"].unique(),
        plt.cm.tab20.colors
    )
})
ax1.barh(top30["feature"], top30["mean_shap"], color=colors.values)
ax1.set_xlabel("Mean |SHAP|")
ax1.set_title("Top 30 Features — Global Importance")
ax1.tick_params(axis="y", labelsize=8)

# ── Plot 2: Group-level importance ────────────────────────────
ax2 = axes[0, 1]
ax2.barh(
    group_importance["group"],
    group_importance["pct"],
    color=plt.cm.tab20.colors[:len(group_importance)]
)
ax2.set_xlabel("% of Total SHAP")
ax2.set_title("Feature Group Importance")
ax2.tick_params(axis="y", labelsize=9)

# ── Plot 3: Fold stability (top 30 by mean rank) ──────────────
ax3 = axes[1, 0]
top30_stable = stability_df.head(30).sort_values("mean_rank", ascending=False)
ax3.barh(
    top30_stable["feature"],
    top30_stable["mean_rank"],
    xerr=top30_stable["std_rank"],
    color="steelblue",
    alpha=0.7,
    error_kw={"ecolor": "red", "capsize": 3},
)
ax3.set_xlabel("Mean Rank (lower = more important)")
ax3.set_title("Top 30 — Fold Stability (error bars = rank std)")
ax3.tick_params(axis="y", labelsize=8)

# ── Plot 4: SHAP beeswarm (built-in shap plot) ────────────────
ax4 = axes[1, 1]
ax4.axis("off")
plt.tight_layout()
plt.savefig("shap_analysis.png", dpi=120, bbox_inches="tight")
plt.show()
print("  Saved: shap_analysis.png")

# Beeswarm in separate figure (shap library manages its own axes)
shap.summary_plot(
    sv,
    X_shap_sample,
    max_display=25,
    show=True,
    plot_title="SHAP Beeswarm — Top 25 Features",
)


# ──────────────────────────────────────────────────────────────
# OUTPUTS — pruned feature lists ready for next step
# ──────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("OUTPUT SUMMARY")
print("=" * 65)
print(f"\n  shap_df          — full feature analysis table ({len(shap_df)} rows)")
print(f"  keep_features    — {len(keep_features)} features to pass to full training")
print(f"  remove_features  — {len(remove_features)} features to drop")
print(f"  review_features  — {len(review_features)} features needing manual check")
print(f"  group_importance — group-level SHAP breakdown")

print(f"\n  To apply pruning to each model split:")
print(f"    X_xgb_pruned = X_xgb_res[keep_features]")
print(f"    X_lgb_pruned = X_lgb_res[keep_features]")
print(f"    X_cgb_pruned = X_cgb_res[keep_features]")
print(f"\n  Then re-run full model training on pruned splits.")
print(f"  Next step: full XGB + LGB + CatBoost training on pruned features.")

In [ ]:
# ──────────────────────────────────────────────────────────────
# STEP 5 — VISUALISATIONS
# ──────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("STEP 5 — SHAP Visualisations")
print("=" * 65)

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle("SHAP Feature Analysis — LightGBM (Light Model)", fontsize=14, y=1.01)

ax1 = axes[0, 0]
top30   = shap_df.head(30).sort_values("mean_shap")
color_map = {g: c for g, c in zip(shap_df["group"].unique(), plt.cm.tab20.colors)}
ax1.barh(top30["feature"], top30["mean_shap"], color=top30["group"].map(color_map).values)
ax1.set_xlabel("Mean |SHAP|"); ax1.set_title("Top 30 Features — Global Importance")
ax1.tick_params(axis="y", labelsize=8)

ax2 = axes[0, 1]
ax2.barh(group_importance["group"], group_importance["pct"],
         color=plt.cm.tab20.colors[:len(group_importance)])
ax2.set_xlabel("% of Total SHAP"); ax2.set_title("Feature Group Importance")
ax2.tick_params(axis="y", labelsize=9)

ax3 = axes[1, 0]
top30_s = stability_df.head(30).sort_values("mean_rank", ascending=False)
ax3.barh(top30_s["feature"], top30_s["mean_rank"], xerr=top30_s["std_rank"],
         color="steelblue", alpha=0.7, error_kw={"ecolor": "red", "capsize": 3})
ax3.set_xlabel("Mean Rank (lower = more important)")
ax3.set_title("Top 30 — Fold Stability (error bars = rank std)")
ax3.tick_params(axis="y", labelsize=8)

axes[1, 1].axis("off")
plt.tight_layout()
plt.savefig("shap_analysis.png", dpi=120, bbox_inches="tight")
plt.show()
print("  Saved: shap_analysis.png")

shap.summary_plot(sv, X_shap_sample, max_display=25, show=True,
                  plot_title="SHAP Beeswarm — Top 25 Features")


# ──────────────────────────────────────────────────────────────
# OUTPUT SUMMARY
# ──────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("OUTPUT SUMMARY")
print("=" * 65)
print(f"\n  shap_df          — full feature analysis table ({len(shap_df)} rows)")
print(f"  keep_features    — {len(keep_features)} features to pass to full training")
print(f"  remove_features  — {len(remove_features)} features to drop")
print(f"  review_features  — {len(review_features)} features needing manual check")
print(f"  group_importance — group-level SHAP breakdown")
print(f"\n  To apply pruning to each model split:")
print(f"    X_xgb_pruned = X_xgb[keep_features]")
print(f"    X_lgb_pruned = X_lgb[keep_features]")
print(f"    X_cgb_pruned = X_cgb[keep_features]")
print(f"\n  Next step: full XGB + LGB + CatBoost training on pruned features.")

In [ ]:

# ================================================================
# FINAL MODEL EVALUATION BLOCK (ROC-AUC + PR-AUC + Threshold Tuning)
# ================================================================

from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, classification_report, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt

def evaluate_model(model, X_train, y_train, X_test, y_test, model_name):
    print("\n" + "="*70)
    print(f"Evaluating {model_name}")
    print("="*70)

    # Train performance
    train_probs = model.predict_proba(X_train)[:, 1]
    train_roc = roc_auc_score(y_train, train_probs)
    train_pr  = average_precision_score(y_train, train_probs)

    # Test performance
    test_probs = model.predict_proba(X_test)[:, 1]
    test_roc = roc_auc_score(y_test, test_probs)
    test_pr  = average_precision_score(y_test, test_probs)

    print(f"Train ROC-AUC: {train_roc:.4f}")
    print(f"Train PR-AUC : {train_pr:.4f}")
    print(f"Test ROC-AUC : {test_roc:.4f}")
    print(f"Test PR-AUC  : {test_pr:.4f}")

    # Precision-Recall Curve
    precision, recall, thresholds = precision_recall_curve(y_test, test_probs)

    plt.figure(figsize=(7, 5))
    plt.plot(recall, precision, label=f'PR Curve (AUC={test_pr:.4f})')
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"{model_name} Precision-Recall Curve")
    plt.legend()
    plt.grid(True)
    plt.show()

    # Threshold tuning using F2-score (recall-focused)
    f2_scores = []
    for t in thresholds:
        preds = (test_probs >= t).astype(int)
        tp = ((preds == 1) & (y_test == 1)).sum()
        fp = ((preds == 1) & (y_test == 0)).sum()
        fn = ((preds == 0) & (y_test == 1)).sum()

        precision_t = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall_t = tp / (tp + fn) if (tp + fn) > 0 else 0

        if precision_t + recall_t == 0:
            f2_scores.append(0)
        else:
            f2 = (5 * precision_t * recall_t) / (4 * precision_t + recall_t)
            f2_scores.append(f2)

    if len(f2_scores) > 0:
        best_idx = np.argmax(f2_scores)
        best_threshold = thresholds[best_idx]
        print(f"Optimal Threshold (F2-score): {best_threshold:.4f}")

        final_preds = (test_probs >= best_threshold).astype(int)

        print("\nConfusion Matrix:")
        print(confusion_matrix(y_test, final_preds))

        print("\nClassification Report:")
        print(classification_report(y_test, final_preds))

    print("="*70)


# Call these after training your models:
# evaluate_model(xgb_model, X_xgb_pruned, y_train, X_xgb_test_pruned, y_test, "XGBoost")
# evaluate_model(lgb_model, X_lgb_pruned, y_train, X_lgb_test_pruned, y_test, "LightGBM")
# evaluate_model(cgb_model, X_cgb_pruned, y_train, X_cgb_test_pruned, y_test, "CatBoost")
